In [ ]:
#STAGES OF INFORMATION VISUALIZATION :
#1 - ACQUIRE - get data from sources 
#2 - PARSE - parse the data with a programming language 
#3 - FILTER - get only the data you need
#4 - MINE - clean and transform them in appropriate data structures 
#5 - REPRESENT - select charts and use data viz libraries 
#6 - REFNE - highlight the take home message
#7 - INTERACT - the environment to present and interact with the results 

<h1>ABSTRACT</h1>

<p>
This project is an exploratory investigation of the ArCo knowledge graph, with a specific focus on historical and artistic cultural heritage assets related to mythology.
</p>

<p>
The objective of the project is to identify and analyze mythological subjects represented within cultural heritage objects preserved in ArCo, exploring how mythological figures, creatures, and narratives appear across different historical periods, artistic contexts, and cultural institutions.
</p>

<p>
Since ArCo does not provide a dedicated controlled vocabulary for mythology-related subjects, Wikidata was used as an external Linked Open Data source to enrich the dataset semantically.
A mythology-related reference dataset was therefore created by extracting entities from Wikidata associated with mythological figures, deities, legendary characters, and other mythology-related categories.
</p>

<p>
The resulting dataset was then used as a semantic reference vocabulary to filter and identify ArCo items whose subjects are connected to mythology.
</p>

COLLECTING MYTHOLOGICAL ENTITIES FROM WIKIDATA
The first phase of the project focused on acquiring a structured dataset of mythology-related entities from Wikidata.
To retrieve data from Wikidata, we connected to the official SPARQL endpoint to exectute SPARQL queries.

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd  

In [ ]:
#richiamo l'endpoint di wikidata 
wikidata_endpoint = "https://query.wikidata.org/sparql"
sparql_wd = SPARQLWrapper(wikidata_endpoint)

To simplify the extraction process, we created a reusable function capable of:
- executing SPARQL queries
- extracting relevant fields from the results
- converting the retrieved data into DataFrames

The function extracts the wikidata entity URI, the label, the aliases of the entity, the descriptions and the semantic categoty associated with the entity.

In [ ]:
#creiamo una funzione per eseguire le query e restituire i risultati in un dataframe

def execute_wikidata_query(query, category):
    sparql_wd.setQuery(query)
    sparql_wd.setReturnFormat(JSON)
    results = sparql_wd.query().convert()
    
    #estraggo i risultati e li metto in un dataframe
    data = []
    for r in results["results"]["bindings"]: #per ogni risultato estraggo i valori dei campi
        entity = r["personaggio"]["value"] if "personaggio" in r else r["evento"]["value"] #se c'è il campo personaggio prendo quello, altrimenti prendo evento

        label = r["persLabel"]["value"] if "persLabel" in r else r["eventLabel"]["value"] #se c'è il campo persLabel prendo quello, altrimenti prendo eventLabel

        aliases = r["aliases"]["value"] if "aliases" in r else None #se c'è il campo aliases prendo quello, altrimenti metto None

        description = r["descrizione"]["value"] if "descrizione" in r else None #se c'è il campo descrizione prendo quello, altrimenti metto None
    
        data.append({
            "entity": entity,
            "label": label,
            "aliases": aliases,
            "description": description,
            "category": category
        })

    df = pd.DataFrame(
        data, 
        columns=["entity", "label", "aliases", "description", "category"]
    )

    return df

Multiple SPARQL queries were created to retrieve different categories of mythology-related entities from Wikidata.

The extracted categories are:
- Greek deity (Q22989102)
- Roman deity (Q11688446)
- Mythological Greek character (Q22988604)
- Mythological Roman character (Q124940323)
- Spirit of nature (Q11879837)
- Mythical war (Q24336068)
- Episode in Greek mythology (Q63143903)
- Mythical entity (Q24334685)
- Class of mythical entities (Q33294038)
- Group of mythical creatures (Q24337101)
- Mythical human-animal hybrid (Q20902363)

In [ ]:
# QUERY PER GREEK DEITIES 
query_greek_deities = """
SELECT ?personaggio ?persLabel
       (GROUP_CONCAT(DISTINCT ?aka; separator=", ") AS ?aliases)
       ?descrizione

WHERE {

  VALUES ?classiAmmesse {
    wd:Q22989102
  }

  ?personaggio wdt:P31/wdt:P279* ?classiAmmesse .

  ?personaggio rdfs:label ?persLabel .
  FILTER(LANG(?persLabel) = "it")

  OPTIONAL {
    ?personaggio skos:altLabel ?aka .
    FILTER(LANG(?aka) = "it")
  }

  OPTIONAL {
    ?personaggio schema:description ?descrizione .
    FILTER(LANG(?descrizione) = "it")
  }

}

GROUP BY ?personaggio ?persLabel ?descrizione
ORDER BY ?persLabel
"""

#DF GREEK DEITIES
df_greek_deities = execute_wikidata_query(query_greek_deities, "greek_deity") 

print(len(df_greek_deities))

In [ ]:
#QUERY PER ROMAN DEITIES 

query_roman_deities = """
SELECT ?personaggio ?persLabel
       (GROUP_CONCAT(DISTINCT ?aka; separator=", ") AS ?aliases)
       ?descrizione

WHERE {

  VALUES ?classiAmmesse {
    wd:Q11688446
  }

  ?personaggio wdt:P31/wdt:P279* ?classiAmmesse .

  ?personaggio rdfs:label ?persLabel .
  FILTER(LANG(?persLabel) = "it")

  OPTIONAL {
    ?personaggio skos:altLabel ?aka .
    FILTER(LANG(?aka) = "it")
  }

  OPTIONAL {
    ?personaggio schema:description ?descrizione .
    FILTER(LANG(?descrizione) = "it")
  }

}

GROUP BY ?personaggio ?persLabel ?descrizione
ORDER BY ?persLabel
"""

#DF ROMAN DEITIES
df_roman_deities = execute_wikidata_query(query_roman_deities, "roman_deity")
print(len(df_roman_deities))

In [ ]:
#QUERY PER GREEK MYTHOLOGICAL CHARACTERS 

query_greek_mythological_characters = """
SELECT ?personaggio ?persLabel
       (GROUP_CONCAT(DISTINCT ?aka; separator=", ") AS ?aliases)
       ?descrizione

WHERE {

  VALUES ?classiAmmesse {
    wd:Q22988604
  }

  ?personaggio wdt:P31/wdt:P279* ?classiAmmesse .

  ?personaggio rdfs:label ?persLabel .
  FILTER(LANG(?persLabel) = "it")

  OPTIONAL {
    ?personaggio skos:altLabel ?aka .
    FILTER(LANG(?aka) = "it")
  }

  OPTIONAL {
    ?personaggio schema:description ?descrizione .
    FILTER(LANG(?descrizione) = "it")
  }

}

GROUP BY ?personaggio ?persLabel ?descrizione
ORDER BY ?persLabel
"""

#DF GREEK MYTHOLOGICAL CHARACTERS
df_greek_mythological_characters = execute_wikidata_query(query_greek_mythological_characters, "greek_mythological_character")
print(len(df_greek_mythological_characters))


In [ ]:
#QUERY PER ROMAN MYTHOLOGICAL CHARACTERS 
query_roman_mythological_characters = """

SELECT ?personaggio ?persLabel
       (GROUP_CONCAT(DISTINCT ?aka; separator=", ") AS ?aliases)
       ?descrizione

WHERE {

  VALUES ?classiAmmesse {
    wd:Q124940323
  }

  ?personaggio wdt:P31/wdt:P279* ?classiAmmesse .

  ?personaggio rdfs:label ?persLabel .
  FILTER(LANG(?persLabel) = "it")

  OPTIONAL {
    ?personaggio skos:altLabel ?aka .
    FILTER(LANG(?aka) = "it")
  }

  OPTIONAL {
    ?personaggio schema:description ?descrizione .
    FILTER(LANG(?descrizione) = "it")
  }

}

GROUP BY ?personaggio ?persLabel ?descrizione
ORDER BY ?persLabel
"""
#DF ROMAN MYTHOLOGICAL CHARACTERS
df_roman_mythological_characters = execute_wikidata_query(query_roman_mythological_characters, "roman_mythological_character")
print(len(df_roman_mythological_characters))

In [ ]:
#QUERY PER GROUP OF DEITIES 
query_group_of_deities = """
SELECT ?personaggio ?persLabel
       (GROUP_CONCAT(DISTINCT ?aka; separator=", ") AS ?aliases)
       ?descrizione

WHERE {

  VALUES ?classiAmmesse {
    wd:Q111252729
  }

  ?personaggio wdt:P31/wdt:P279* ?classiAmmesse .

  ?personaggio rdfs:label ?persLabel .
  FILTER(LANG(?persLabel) = "it")

  OPTIONAL {
    ?personaggio skos:altLabel ?aka .
    FILTER(LANG(?aka) = "it")
  }

  OPTIONAL {
    ?personaggio schema:description ?descrizione .
    FILTER(LANG(?descrizione) = "it")
  }

  FILTER(
    BOUND(?descrizione) &&
    REGEX(LCASE(?descrizione), "greek|roman|greca|romana")
  )

}

GROUP BY ?personaggio ?persLabel ?descrizione
ORDER BY ?persLabel
"""

#DF GROUP OF DEITIES
df_group_of_deities = execute_wikidata_query(query_group_of_deities, "group_of_deities")
print(len(df_group_of_deities))


In [ ]:
#QUERY PER SPIRIT OF NATURE 

query_spirits_of_nature = """
SELECT ?personaggio ?persLabel
       (GROUP_CONCAT(DISTINCT ?aka; separator=", ") AS ?aliases)
       ?descrizione

WHERE {

  VALUES ?classiAmmesse {
    wd:Q11879837
  }

  ?personaggio wdt:P31/wdt:P279* ?classiAmmesse .

  FILTER NOT EXISTS {
    ?personaggio wdt:P31 ?excluded .
    VALUES ?excluded {
      wd:Q30168533
      wd:Q15711870
      wd:Q15632617
      wd:Q15773347
      wd:Q1114461
      wd:Q87576284
      wd:Q80447738
    }
  }

  ?personaggio rdfs:label ?persLabel .
  FILTER(LANG(?persLabel) = "it")

  OPTIONAL {
    ?personaggio skos:altLabel ?aka .
    FILTER(LANG(?aka) = "it")
  }

  OPTIONAL {
    ?personaggio schema:description ?descrizione .
    FILTER(LANG(?descrizione) = "it")
  }

  FILTER(
    !BOUND(?descrizione) ||
    REGEX(LCASE(?descrizione), "grec|roman")
  )

}

GROUP BY ?personaggio ?persLabel ?descrizione
ORDER BY ?persLabel
"""
df_spirits_of_nature = execute_wikidata_query(query_spirits_of_nature, "spirits_of_nature")
print(len(df_spirits_of_nature))

In [ ]:
#QUERY PER MYTHICAL WAR 

query_mythical_war = """
SELECT ?evento ?eventLabel
       (GROUP_CONCAT(DISTINCT ?aka; separator=", ") AS ?aliases)
       ?descrizione

WHERE {

  VALUES ?classiAmmesse {
    wd:Q24336068
    wd:Q63143903
  }

  VALUES ?mitiAmmessi {
    wd:Q34726
    wd:Q122173
  }

  ?evento wdt:P31/wdt:P279* ?classiAmmesse ;
          wdt:P361 ?mitiAmmessi .

  ?evento rdfs:label ?eventLabel .
  FILTER(LANG(?eventLabel) = "it")

  OPTIONAL {
    ?evento skos:altLabel ?aka .
    FILTER(LANG(?aka) = "it")
  }

  OPTIONAL {
    ?evento schema:description ?descrizione .
    FILTER(LANG(?descrizione) = "it")
  }

}

GROUP BY ?evento ?eventLabel ?descrizione
ORDER BY ?eventLabel
"""
df_mythical_war = execute_wikidata_query(query_mythical_war, "mythical_war")
print(len(df_mythical_war))


In [ ]:
#QUERY PER MYTICAL ENTITIES/ENTITY/GROUP OF MYTHICAL CREATURES 

query_mythical_entities = """
SELECT ?personaggio ?persLabel
       (GROUP_CONCAT(DISTINCT ?aka; separator=", ") AS ?aliases)
       ?descrizione

WHERE {

  VALUES ?classiAmmesse {
    wd:Q24334685
    wd:Q33294038
    wd:Q24337101
  }

  VALUES ?mitiAmmessi {
    wd:Q34726
    wd:Q122173
  }

  ?personaggio wdt:P31/wdt:P279* ?classiAmmesse ;
              wdt:P361 ?mitiAmmessi .

  ?personaggio rdfs:label ?persLabel .
  FILTER(LANG(?persLabel) = "it")

  OPTIONAL {
    ?personaggio skos:altLabel ?aka .
    FILTER(LANG(?aka) = "it")
  }

  OPTIONAL {
    ?personaggio schema:description ?descrizione .
    FILTER(LANG(?descrizione) = "it")
  }

}

GROUP BY ?personaggio ?persLabel ?descrizione
ORDER BY ?persLabel
"""
df_mythical_entities = execute_wikidata_query(query_mythical_entities, "mythical_entity")
print(len(df_mythical_entities))


In [ ]:
#QUERY PER IBRIDO 
query_ibrido = """
SELECT ?personaggio ?persLabel
       (GROUP_CONCAT(DISTINCT ?aka; separator=", ") AS ?aliases)
       ?descrizione

WHERE {

  VALUES ?classiAmmesse {
    wd:Q20902363
  }

  ?personaggio wdt:P31/wdt:P279* ?classiAmmesse .

  FILTER NOT EXISTS {
    ?personaggio wdt:P31/wdt:P279* ?excludedClass .
    VALUES ?excludedClass {
      wd:Q30168533
      wd:Q15711870
      wd:Q15632617
      wd:Q15773347
      wd:Q1114461
      wd:Q87576284
      wd:Q80447738
    }
  }

  ?personaggio rdfs:label ?persLabel .
  FILTER(LANG(?persLabel) = "it")

  OPTIONAL {
    ?personaggio skos:altLabel ?aka .
    FILTER(LANG(?aka) = "it")
  }

  OPTIONAL {
    ?personaggio schema:description ?descrizione .
    FILTER(LANG(?descrizione) = "it")
  }

}

GROUP BY ?personaggio ?persLabel ?descrizione
ORDER BY ?persLabel
"""
df_ibrido = execute_wikidata_query(query_ibrido, "ibrido")
df_ibrido = df_ibrido[df_ibrido["label"] != "Sirenetta"]  #ho tolto la sirenetta   
print(len(df_ibrido))

After executing all SPARQL queries, the resulting DataFrames were merged into a single dataset.

In [ ]:
#ORA CREIAMO UN UNICO DATAFRAME CON TUTTE LE CATEGORIE SENZA I DUPLICATI

df_uniti = pd.concat([
    df_greek_deities,
    df_roman_deities,
    df_greek_mythological_characters,
    df_roman_mythological_characters,
    df_group_of_deities,
    df_spirits_of_nature,
    df_mythical_war,
    df_mythical_entities,
    df_ibrido
], ignore_index=True)

len(df_uniti)
#df_uniti.head(1)



Before exporting the dataset, a cleaning phase was performed to improve data quality.

First, entities without descriptions were removed.

In [ ]:
#rimuovo le entità senza descrizione o con descrizione vuota
df_uniti = df_uniti[
    df_uniti["description"].notna() &
    (df_uniti["description"].astype(str).str.strip() != "")
].copy()

len(df_uniti)

Duplicate entities were then removed using the Wikidata URI as a unique identifier.

In [ ]:
#elimino le entità duplicate
df_final = df_uniti.drop_duplicates(subset="entity")
len(df_final)

In [ ]:
#vediamo tutto il dataframe finale
df_final.sort_values("label")

The final mythology dataset was exported as a CSV file for subsequent stages of the workflow. 
The resulting CSV file constitutes the semantic reference vocabulary used in the following stages of the project to identify mythology-related cultural heritage items within the ArCo knowledge graph.

In [ ]:
#esporto il dataframe in un file csv
wikidata_csv_updated = df_final.to_csv("wd_dataset.csv", index=False, sep=",", encoding="utf-8")